# MLTrader: Strategy Analysis & Lessons Learned

This notebook documents the development of MLTrader — a hybrid ML + LLM paper trading bot built as a continuation of Georgia Tech's CS 7646: Machine Learning for Trading.

It walks through:
1. The initial approach and why it failed
2. Each improvement made and the reasoning behind it
3. Before/after backtest results on SPY and NVDA
4. Key takeaways and future directions

> **Note:** All backtests use a strict out-of-sample test window (most recent year) with the model trained on the prior 2 years. No lookahead bias.

In [ ]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from StrategyLearner import StrategyLearner
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
import config

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

# ── Date windows (all relative to today) ──────────────────────────────────────
now         = datetime.now(ZoneInfo('America/New_York'))
TRAIN_START = now - timedelta(days=365 * 3)
TRAIN_END   = now - timedelta(days=365)
TEST_START  = TRAIN_END
TEST_END    = now - timedelta(days=1)

# ── Alpaca data client ────────────────────────────────────────────────────────
_client = StockHistoricalDataClient(config.public, config.secret)

def fetch_prices(symbol):
    bars = _client.get_stock_bars(StockBarsRequest(
        symbol_or_symbols=[symbol], timeframe=TimeFrame.Day,
        start=TEST_START, end=TEST_END
    )).df
    p = bars['close'].reset_index(level='symbol', drop=True)
    p.index = p.index.tz_localize(None)
    return p

print('Setup complete.')

In [ ]:
def backtest(trades, prices, label='Strategy'):
    """Simulate portfolio value from a trade signal DataFrame."""
    p          = prices.reindex(trades.index).ffill()
    daily_ret  = p.pct_change().fillna(0)
    # Use previous day's signal to avoid lookahead bias
    signal_dir = trades['shares'].shift(1).fillna(0).apply(
        lambda x: 1 if x > 0 else (-1 if x < 0 else 0)
    )
    strat_ret = signal_dir * daily_ret
    strat_val = (1 + strat_ret).cumprod()
    bah_val   = (1 + daily_ret).cumprod()
    roll_max  = strat_val.cummax()
    mdd       = float(((strat_val - roll_max) / roll_max).min() * 100)

    return {
        'label':    label,
        'ret':      float(strat_val.iloc[-1] - 1) * 100,
        'bah':      float(bah_val.iloc[-1]   - 1) * 100,
        'mdd':      mdd,
        'days_in':  int((signal_dir != 0).sum()),
        'total':    len(trades),
        'buys':     int((trades['shares'] > 0).sum()),
        'sells':    int((trades['shares'] < 0).sum()),
        'curve':    strat_val,
        'bah_curve':bah_val,
    }


def print_results(r):
    print(f"  Return       : {r['ret']:+.1f}%")
    print(f"  Buy-and-hold : {r['bah']:+.1f}%")
    print(f"  vs B&H       : {r['ret']-r['bah']:+.1f}%")
    print(f"  Max drawdown : {r['mdd']:.1f}%")
    print(f"  Days in mkt  : {r['days_in']}/{r['total']} ({r['days_in']/r['total']*100:.0f}%)")
    print(f"  Signals      : {r['buys']} buys / {r['sells']} sells")


def plot_comparison(results_list, prices, symbol, title):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=13, fontweight='bold')

    # Left: portfolio value curves
    ax = axes[0]
    ax.set_title('Portfolio Value (starting $1.00)')
    bah_plotted = False
    for r in results_list:
        ax.plot(r['curve'].index, r['curve'].values, label=r['label'], linewidth=2)
        if not bah_plotted:
            ax.plot(r['bah_curve'].index, r['bah_curve'].values,
                    label='Buy & Hold', linewidth=1.5, linestyle='--', color='gray')
            bah_plotted = True
    ax.axhline(1.0, color='black', linewidth=0.5, linestyle=':')
    ax.set_xlabel('Date')
    ax.set_ylabel('Portfolio value')
    ax.legend()
    plt.setp(ax.get_xticklabels(), rotation=30)

    # Right: summary bar chart
    ax2 = axes[1]
    ax2.set_title('Total Return Comparison')
    labels = [r['label'] for r in results_list] + ['Buy & Hold']
    values = [r['ret'] for r in results_list] + [results_list[0]['bah']]
    colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in values]
    colors[-1] = '#95a5a6'  # grey for B&H
    bars = ax2.bar(labels, values, color=colors, edgecolor='white', linewidth=0.5)
    ax2.axhline(0, color='black', linewidth=0.5)
    for bar, val in zip(bars, values):
        ax2.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + (1 if val >= 0 else -3),
                 f'{val:+.1f}%', ha='center', va='bottom', fontsize=9)
    ax2.set_ylabel('Return (%)')
    ax2.set_xticklabels(labels, rotation=15)

    plt.tight_layout()
    plt.show()

print('Helper functions ready.')

---
## Part 1 — The Initial Approach

The original strategy came directly from GT CS 7646 Project 8 (Strategy Learner). The approach:

- **Model:** Bagged Random Trees (100 bags, leaf_size=5)
- **Features:** SMA ratio, Bollinger %B, Momentum, PPO histogram
- **Labels:** +1 (buy), -1 (sell), 0 (hold) based on a **5-day** future return vs a 0.5% threshold
- **Default position:** Flat (out of market) — only enter on a buy signal
- **Shorting:** Allowed on sell signals

This was tested on **SPY** (S&P 500 ETF) over a 1-year out-of-sample window.

In [ ]:
print('Training original strategy on SPY...')
spy_prices = fetch_prices('SPY')

old_spy = StrategyLearner(verbose=False, impact=0.005)
old_spy.add_evidence('SPY', sd=TRAIN_START, ed=TRAIN_END, lookahead=5)
old_trades_spy = old_spy.testPolicy('SPY', sd=TEST_START, ed=TEST_END,
                                     long_bias=False, lookahead=5)

r_old_spy = backtest(old_trades_spy, spy_prices, label='Original (5-day)')
print('\nOriginal strategy on SPY:')
print_results(r_old_spy)

### What went wrong?

The strategy massively underperformed buy-and-hold. The root causes:

| Problem | Impact |
|---|---|
| **Mostly out of market** | Only ~16-26% of days invested — missed most of SPY's gains |
| **5-day labels are pure noise** | Daily returns are nearly random over 5 days — the model learned nothing real |
| **Shorting SPY** | Betting against the S&P 500's natural upward drift is a losing strategy |
| **Balanced label training** | Treating buy/hold/sell as equally likely ignores the market's long-run upward bias |
| **LLM required to agree** | Made the system even more conservative — almost never fired |

The key insight: **for an index ETF that trends upward over time, time in market almost always beats timing the market.**

---
## Part 2 — The Improvements

Five targeted fixes were made. Each addresses a specific problem identified above.

### Lesson 1: Extend the label lookahead (5 days → 20 days)

**Problem:** 5-day future returns are dominated by random noise. The model was learning patterns that don't generalise.

**Fix:** Changed the lookahead from 5 to 20 trading days (~1 month). This:
- Captures actual medium-term trends instead of daily noise
- Produces more balanced buy/sell labels (more days move >0.5% over 20 days than over 5)
- Makes learned patterns more durable

The cell below shows how the label distribution changes:

In [ ]:
learner_tmp = StrategyLearner(verbose=False, impact=0.005)
_, prices_tmp = learner_tmp._fetch_indicators('SPY', sd=TRAIN_START, ed=TRAIN_END)

labels_5  = learner_tmp._make_labels(prices_tmp, lookahead=5)
labels_20 = learner_tmp._make_labels(prices_tmp, lookahead=20)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Label Distribution: SPY Training Data', fontweight='bold')

for ax, labels, title in [
    (axes[0], labels_5,  '5-day lookahead (original)'),
    (axes[1], labels_20, '20-day lookahead (improved)'),
]:
    counts = labels.value_counts().sort_index()
    bars = ax.bar(['-1 (Sell)', '0 (Hold)', '+1 (Buy)'],
                  [counts.get(-1.0,0), counts.get(0.0,0), counts.get(1.0,0)],
                  color=['#e74c3c','#95a5a6','#2ecc71'])
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+1,
                str(int(b.get_height())), ha='center', fontsize=9)
    ax.set_title(title)
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

### Lesson 2: Long bias for ETFs (default to invested)

**Problem:** Starting flat and waiting for a buy signal means sitting in cash while the market rises.

**Fix:** For broad-market ETFs (SPY, QQQ, IWM, etc.), start **long by default**. The model only exits on a confirmed sell signal and re-enters on a buy signal.

This changes the question from *"when should I buy?"* to *"when should I step aside?"* — a much better framing for assets with long-run upward drift.

```python
LONG_BIAS_SYMBOLS = {"SPY", "QQQ", "IWM", "DIA", "VTI", "VOO", "IVV", "GLD"}
```

### Lesson 3: Two new features — ATR% and Volume Ratio

**Problem:** The 4 original features (SMA, BBP, Momentum, PPO) only describe *price direction*. They give no information about *conviction* behind moves.

**Fix:** Added two confirming features:

| Feature | What it measures | Why it helps |
|---|---|---|
| **ATR%** (14-day Average True Range / price) | Volatility normalised to price level | High volatility = larger moves expected; helps model be more selective |
| **Volume Ratio** (today / 20-day avg) | Whether today's volume is above or below average | >1.0 = strong conviction behind price move; <1.0 = weak signal |

Together these add context that pure price-based indicators miss.

### Lesson 4: Never short broad-market ETFs

**Problem:** The original model generated ~20 sell signals per year on SPY, effectively shorting the S&P 500. The market has a long-run upward bias — fighting that is a structural losing bet.

**Fix:** Added a `LONG_BIAS_SYMBOLS` blocklist. For any symbol in this set, sell signals exit to flat rather than going short.

Shorting is still allowed for individual stocks (e.g. NVDA, AAPL) where negative trends can be sustained.

### Lesson 5: LLM as a veto, not a co-requirer

**Problem:** The original rule required *both* ML and LLM to agree before acting. Since the ML model was already conservative, adding an LLM agreement requirement made it almost impossible to ever trade.

**Fix:** Flipped the logic — the ML model drives all signals. The LLM only **blocks** a trade if it strongly disagrees (opposite signal, confidence ≥ 0.70).

```
OLD: ML says buy AND LLM says buy AND confidence >= 0.60  →  execute
NEW: ML says buy UNLESS LLM says sell AND confidence >= 0.70  →  execute
```

This preserves the LLM's value as a safety net against bad ML signals while not throttling every trade.

---
## Part 3 — Results: SPY Before vs After

In [ ]:
print('Training improved strategy on SPY...')
new_spy = StrategyLearner(verbose=False, impact=0.005)
new_spy.add_evidence('SPY', sd=TRAIN_START, ed=TRAIN_END, lookahead=20)
new_trades_spy = new_spy.testPolicy('SPY', sd=TEST_START, ed=TEST_END,
                                     long_bias=True, lookahead=20)

r_new_spy = backtest(new_trades_spy, spy_prices, label='Improved (20-day, long bias)')

print('\n--- SPY Original ---')
print_results(r_old_spy)
print('\n--- SPY Improved ---')
print_results(r_new_spy)
print(f'\nImprovement: {r_new_spy["ret"]-r_old_spy["ret"]:+.1f}%')

plot_comparison([r_old_spy, r_new_spy], spy_prices, 'SPY',
                'SPY: Original vs Improved Strategy (1-year out-of-sample)')

---
## Part 4 — Applying to NVDA

SPY's relentless upward trend makes it very hard to beat buy-and-hold with any timing model. Individual stocks are a better test case — they have:
- Higher volatility (more signal in technical indicators)
- Clearer trend cycles driven by earnings, product cycles, sector sentiment
- Meaningful downside periods where exiting/shorting adds real value

NVDA (Nvidia) is tested here — driven by the AI cycle, it has had massive moves in both directions.

In [ ]:
print('Fetching NVDA prices...')
nvda_prices = fetch_prices('NVDA')

print('Training original strategy on NVDA...')
old_nvda = StrategyLearner(verbose=False, impact=0.005)
old_nvda.add_evidence('NVDA', sd=TRAIN_START, ed=TRAIN_END, lookahead=5)
old_trades_nvda = old_nvda.testPolicy('NVDA', sd=TEST_START, ed=TEST_END,
                                       long_bias=False, lookahead=5)

print('Training improved strategy on NVDA...')
new_nvda = StrategyLearner(verbose=False, impact=0.005)
new_nvda.add_evidence('NVDA', sd=TRAIN_START, ed=TRAIN_END, lookahead=20)
new_trades_nvda = new_nvda.testPolicy('NVDA', sd=TEST_START, ed=TEST_END,
                                       long_bias=False, lookahead=20)

r_old_nvda = backtest(old_trades_nvda, nvda_prices, label='Original (5-day)')
r_new_nvda = backtest(new_trades_nvda, nvda_prices, label='Improved (20-day)')

print('\n--- NVDA Original ---')
print_results(r_old_nvda)
print('\n--- NVDA Improved ---')
print_results(r_new_nvda)
print(f'\nImprovement: {r_new_nvda["ret"]-r_old_nvda["ret"]:+.1f}%')

plot_comparison([r_old_nvda, r_new_nvda], nvda_prices, 'NVDA',
                'NVDA: Original vs Improved Strategy (1-year out-of-sample)')

---
## Part 5 — Summary Table

In [ ]:
summary = pd.DataFrame([
    {
        'Symbol': 'SPY',
        'Strategy': 'Original (5-day, flat default)',
        'Return': f"{r_old_spy['ret']:+.1f}%",
        'Buy-and-Hold': f"{r_old_spy['bah']:+.1f}%",
        'vs B&H': f"{r_old_spy['ret']-r_old_spy['bah']:+.1f}%",
        'Max Drawdown': f"{r_old_spy['mdd']:.1f}%",
        'Days in Market': f"{r_old_spy['days_in']}/{r_old_spy['total']} ({r_old_spy['days_in']/r_old_spy['total']*100:.0f}%)",
    },
    {
        'Symbol': 'SPY',
        'Strategy': 'Improved (20-day, long bias)',
        'Return': f"{r_new_spy['ret']:+.1f}%",
        'Buy-and-Hold': f"{r_new_spy['bah']:+.1f}%",
        'vs B&H': f"{r_new_spy['ret']-r_new_spy['bah']:+.1f}%",
        'Max Drawdown': f"{r_new_spy['mdd']:.1f}%",
        'Days in Market': f"{r_new_spy['days_in']}/{r_new_spy['total']} ({r_new_spy['days_in']/r_new_spy['total']*100:.0f}%)",
    },
    {
        'Symbol': 'NVDA',
        'Strategy': 'Original (5-day, flat default)',
        'Return': f"{r_old_nvda['ret']:+.1f}%",
        'Buy-and-Hold': f"{r_old_nvda['bah']:+.1f}%",
        'vs B&H': f"{r_old_nvda['ret']-r_old_nvda['bah']:+.1f}%",
        'Max Drawdown': f"{r_old_nvda['mdd']:.1f}%",
        'Days in Market': f"{r_old_nvda['days_in']}/{r_old_nvda['total']} ({r_old_nvda['days_in']/r_old_nvda['total']*100:.0f}%)",
    },
    {
        'Symbol': 'NVDA',
        'Strategy': 'Improved (20-day, 6 features)',
        'Return': f"{r_new_nvda['ret']:+.1f}%",
        'Buy-and-Hold': f"{r_new_nvda['bah']:+.1f}%",
        'vs B&H': f"{r_new_nvda['ret']-r_new_nvda['bah']:+.1f}%",
        'Max Drawdown': f"{r_new_nvda['mdd']:.1f}%",
        'Days in Market': f"{r_new_nvda['days_in']}/{r_new_nvda['total']} ({r_new_nvda['days_in']/r_new_nvda['total']*100:.0f}%)",
    },
])

pd.set_option('display.max_colwidth', None)
summary

---
## Key Takeaways

### 1. Time in market beats timing the market (for indexes)
A strategy that is only 16-26% invested will almost always underperform a broad ETF that trends up 20-25% per year. For SPY, the best technical model is a very high bar to clear.

### 2. Label quality matters more than model complexity
The model architecture (Bagged Random Trees) was never the bottleneck. Switching from 5-day to 20-day labels — a one-line change — had more impact than any model tuning could.

### 3. Domain knowledge > raw ML
Knowing that SPY trends up and that shorting it is structurally bad (Lessons 2 and 4) added more value than adding features or tuning hyperparameters. Understanding *what* you're trading is as important as *how* you model it.

### 4. LLMs add qualitative judgment, not quantitative precision
The Gemini research layer correctly identified market context (near upper Bollinger Band, mixed news). Its value is as a *sanity check* on ML signals, not as a precise signal generator. The veto-only design reflects this.

### 5. Individual stocks are a better proving ground
NVDA's +30% improvement (old → new) vs SPY's +13.5% confirms that technical indicators work better on volatile individual stocks than on smooth index ETFs.

---
## Future Directions

| Idea | Expected Impact |
|---|---|
| **Regime detection** (bull/bear/sideways) | Don't apply the same model to all market conditions |
| **Confidence-weighted position sizing** | Scale position size with signal confidence instead of binary in/out |
| **Walk-forward validation** | Retrain on a rolling window to avoid model staleness |
| **Earnings calendar awareness** | Avoid holding through earnings unless specifically trading earnings |
| **Multi-symbol portfolio** | Diversify signals across several uncorrelated names |
| **Finer LLM tool use** | Give Gemini access to earnings transcripts, analyst ratings, options flow |